# 13A Appendix: PCA as an Eigenproblem, and the t-SNE Objective


The derivations behind the results stated in [Lecture 13](13-Dimensionality-Reduction-PCA-tSNE.qmd). Nothing here is needed to *use* PCA or t-SNE, and none of it is assessed; it is for the reader who wants to know why the eigenvectors of the covariance matrix are the right directions, why the SVD of the centered data is the same computation, and what t-SNE is actually minimizing.

Throughout, $X_0 \in \mathbb{R}^{m\times n}$ holds $m$ points as rows and $n$ features as columns, and $X$ is the column-centered version.

## Why $S$ Is the Covariance Matrix

The lecture states that for centered data

$$
S = \frac{1}{m-1}X^{T}X
$$

is the sample covariance matrix. Write the columns of $X$ as vectors,

$$
X = \begin{bmatrix} | & | & & | \\ \mathbf{x}_1 & \mathbf{x}_2 & \cdots & \mathbf{x}_n \\ | & | & & | \end{bmatrix},
$$

where each $\mathbf{x}_j \in \mathbb{R}^m$ is one feature across all $m$ observations, with mean zero because the data is centered.

The product $X^TX$ is $(n \times m)(m \times n) = n \times n$, and its $(i,j)$ entry is the dot product of column $i$ with column $j$:

$$
(X^TX)_{ij} = \mathbf{x}_i^T\mathbf{x}_j = \sum_{k=1}^{m} x_{ki}\, x_{kj}.
$$

The sample covariance between features $i$ and $j$ is defined as

$$
\operatorname{Cov}(i,j) = \frac{1}{m-1}\sum_{k=1}^{m}(x_{ki} - \bar{x}_i)(x_{kj} - \bar{x}_j),
$$

and since the data is centered, $\bar{x}_i = \bar{x}_j = 0$, so

$$
\operatorname{Cov}(i,j) = \frac{1}{m-1}\sum_{k=1}^{m}x_{ki}\, x_{kj} = \frac{1}{m-1}(X^TX)_{ij}.
$$

That is exactly the $(i,j)$ entry of $S$. Its diagonal entries $S_{ii}$ are the variances of the features and its off-diagonal entries $S_{ij}$ are their covariances. The factor $m-1$ instead of $m$ is Bessel's correction, which makes $S$ the unbiased estimator; nothing below depends on the choice.

Centering is what makes this work. On uncentered data, $\frac{1}{m-1}X_0^TX_0$ is dominated by the squared means, and the leading "principal component" points from the origin toward the centroid of the cloud instead of along its direction of greatest spread.

## The Maximum-Variance Direction Is the Top Eigenvector

### Variance along a direction

Pick any direction: a unit vector $\mathbf{w}\in\mathbb{R}^n$ with $\|\mathbf{w}\| = 1$. Projecting every centered point onto it gives one coordinate per point, the vector $X\mathbf{w}\in\mathbb{R}^m$, which has mean zero because the columns of $X$ do. Its variance is

$$
\operatorname{Var}(X\mathbf{w}) = \frac{1}{m-1}(X\mathbf{w})^{T}(X\mathbf{w}) = \mathbf{w}^{T}\left(\frac{1}{m-1}X^{T}X\right)\mathbf{w} = \mathbf{w}^{T}S\mathbf{w}.
$$

So "find the direction of maximum variance" is the constrained problem

$$
\max_{\mathbf{w}} \; \mathbf{w}^{T}S\mathbf{w} \quad \text{subject to} \quad \mathbf{w}^{T}\mathbf{w} = 1.
$$

On the lecture's example, $S = \begin{bmatrix} 3 & 1.5 \\ 1.5 & 3 \end{bmatrix}$: the direction $\mathbf{w} = \begin{bmatrix}1 & 0\end{bmatrix}^T$ (feature 1 alone) gives $\mathbf{w}^TS\mathbf{w} = 3$, the diagonal $\frac{1}{\sqrt{2}}\begin{bmatrix}1 & 1\end{bmatrix}^T$ gives $\frac{1}{2}(3 + 1.5 + 1.5 + 3) = 4.5$, and the other diagonal $\frac{1}{\sqrt{2}}\begin{bmatrix}1 & -1\end{bmatrix}^T$ gives $1.5$.

### The Lagrangian

Introduce a multiplier $\lambda$ for the length constraint and set the gradient of

$$
\mathcal{L}(\mathbf{w}, \lambda) = \mathbf{w}^{T}S\mathbf{w} - \lambda(\mathbf{w}^{T}\mathbf{w} - 1)
$$

with respect to $\mathbf{w}$ to zero. Because $S$ is symmetric, $\nabla_{\mathbf{w}}(\mathbf{w}^TS\mathbf{w}) = 2S\mathbf{w}$, so

$$
2S\mathbf{w} - 2\lambda\mathbf{w} = \mathbf{0} \quad\Longrightarrow\quad S\mathbf{w} = \lambda\mathbf{w}.
$$

Every stationary point of the constrained problem is an eigenvector of $S$. At such a point the objective equals the eigenvalue:

$$
\mathbf{w}^{T}S\mathbf{w} = \mathbf{w}^{T}(\lambda\mathbf{w}) = \lambda\,\mathbf{w}^{T}\mathbf{w} = \lambda.
$$

The maximum is therefore attained at the eigenvector with the largest eigenvalue, $\lambda_1$, and the variance it achieves is $\lambda_1$ itself.

### The Rayleigh quotient

The same conclusion without the multiplier: for any nonzero $\mathbf{w}$, the ratio

$$
R(\mathbf{w}) = \frac{\mathbf{w}^TS\mathbf{w}}{\mathbf{w}^T\mathbf{w}}
$$

is the *Rayleigh quotient* of $S$. Expand $\mathbf{w}$ in the orthonormal eigenbasis, $\mathbf{w} = \sum_i c_i\mathbf{v}_i$. Then $\mathbf{w}^TS\mathbf{w} = \sum_i \lambda_i c_i^2$ and $\mathbf{w}^T\mathbf{w} = \sum_i c_i^2$, so $R(\mathbf{w})$ is a weighted average of the eigenvalues with non-negative weights $c_i^2 / \sum_k c_k^2$. A weighted average of numbers is at most the largest of them, with equality when all the weight sits on $\lambda_1$, that is, when $\mathbf{w} \propto \mathbf{v}_1$. So

$$
\lambda_n \leq R(\mathbf{w}) \leq \lambda_1 \quad \text{for every } \mathbf{w} \neq \mathbf{0}.
$$

### The remaining directions

For the second principal direction, add the constraint $\mathbf{w} \perp \mathbf{v}_1$. In the eigenbasis that sets $c_1 = 0$, and the same weighted-average argument gives a maximum of $\lambda_2$ at $\mathbf{w} = \mathbf{v}_2$. Repeating, the $k$-th principal direction is the eigenvector $\mathbf{v}_k$ with the $k$-th largest eigenvalue, constrained to be orthogonal to the first $k-1$. This is why the spectral decomposition $S = V\Lambda V^T$, with the eigenvalues sorted in decreasing order, hands over all the principal directions at once.

Two facts the lecture uses follow immediately:

- The columns of $Y = XV'$ have variances $\lambda_1, \dots, \lambda_d$, since column $j$ is $X\mathbf{v}_j$ and $\operatorname{Var}(X\mathbf{v}_j) = \mathbf{v}_j^TS\mathbf{v}_j = \lambda_j$.
- Those columns are uncorrelated: their covariance is $\frac{1}{m-1}(X\mathbf{v}_i)^T(X\mathbf{v}_j) = \mathbf{v}_i^TS\mathbf{v}_j = \lambda_j\mathbf{v}_i^T\mathbf{v}_j = 0$ for $i \neq j$.
- The eigenvalues are non-negative, because $\mathbf{w}^TS\mathbf{w} = \frac{1}{m-1}\|X\mathbf{w}\|^2 \geq 0$ for every $\mathbf{w}$; a covariance matrix is positive semidefinite.

### Maximum variance equals minimum residual

The lecture's two pictures are the same optimization. For a point $\mathbf{x}_i$ and a unit direction $\mathbf{w}$, the projection onto the line is $(\mathbf{x}_i\cdot\mathbf{w})\mathbf{w}$ and the residual is the perpendicular remainder, so by Pythagoras

$$
\|\mathbf{x}_i\|^2 = (\mathbf{x}_i\cdot\mathbf{w})^2 + \|\mathbf{x}_i - (\mathbf{x}_i\cdot\mathbf{w})\mathbf{w}\|^2.
$$

Summing over the points,

$$
\underbrace{\sum_i \|\mathbf{x}_i\|^2}_{\text{fixed}} = \underbrace{\sum_i (\mathbf{x}_i\cdot\mathbf{w})^2}_{(m-1)\operatorname{Var}(X\mathbf{w})} + \underbrace{\sum_i \text{residual}_i^2}_{\text{squared residuals}}.
$$

The left side does not depend on $\mathbf{w}$, so maximizing the projected variance is the same as minimizing the sum of squared residuals. The same identity, applied to all $d$ directions at once, is what makes the reconstruction error below equal to the discarded variance.

## PCA Is the SVD of the Centered Data

Let the thin SVD of the centered data be $X = U\Sigma V^T$, with $U \in \mathbb{R}^{m\times n}$ having orthonormal columns, $\Sigma = \operatorname{diag}(\sigma_1, \dots, \sigma_n)$ with $\sigma_1 \geq \dots \geq \sigma_n \geq 0$, and $V \in \mathbb{R}^{n\times n}$ orthogonal.

### The eigendecomposition of $X^TX$

$$
X^{T}X = (U\Sigma V^T)^{T}(U\Sigma V^T) = V\Sigma U^TU \Sigma V^{T} = V\Sigma^{2}V^T,
$$

using $U^TU = I$. This is a spectral decomposition of the symmetric matrix $X^TX$: orthonormal eigenvectors $V$ and eigenvalues $\sigma_i^2$. Dividing by $m-1$,

$$
S = \frac{1}{m-1}X^TX = V\left(\frac{\Sigma^2}{m-1}\right)V^T,
$$

so the principal directions are the right singular vectors $V$ and $\lambda_i = \sigma_i^2/(m-1)$. The eigendecomposition is unique up to the sign of each eigenvector (and the choice of basis within a repeated eigenvalue), which is why `numpy.linalg.eigh`, `numpy.linalg.svd` and scikit-learn's `PCA` are each free to return $\mathbf{v}$ or $-\mathbf{v}$.

### Scores

Let $V'$ be the first $d$ columns of $V$, $U'$ the first $d$ columns of $U$, and $\Sigma'$ the top-left $d\times d$ block of $\Sigma$. Then

$$
Y = XV' = U\Sigma V^TV' = U'\Sigma',
$$

because $V^TV'$ is the first $d$ columns of the $n\times n$ identity. Column $j$ of $Y$ is $\sigma_j\mathbf{u}_j$; since $\|\mathbf{u}_j\| = 1$ its variance is $\sigma_j^2/(m-1) = \lambda_j$, as the eigen-argument promised.

### Reconstruction and the discarded variance

Mapping the scores back to $\mathbb{R}^n$,

$$
X^{(d)} = YV'^T = U'\Sigma'V'^T,
$$

which is exactly the rank-$d$ truncated SVD of $X$. By the Eckart--Young theorem (lecture 12) it is the best rank-$d$ approximation of $X$ in the Frobenius norm. PCA with $d$ components and the rank-$d$ truncated SVD of the *centered* data are the same thing.

The error of that approximation is

$$
\|X - X^{(d)}\|_F^2 = \sum_{i>d}\sigma_i^2 = (m-1)\sum_{i>d}\lambda_i,
$$

the variance in the directions that were dropped. Dividing by $\|X\|_F^2 = \sum_i\sigma_i^2 = (m-1)T$ gives

$$
\frac{\|X - X^{(d)}\|_F^2}{\|X\|_F^2} = 1 - \sum_{i=1}^{d}\frac{\lambda_i}{T},
$$

one minus the cumulative explained variance through $d$. Keep 90% of the variance and you accept a 10% squared reconstruction error.

### Why practice uses the SVD

- **Numerical stability.** Forming $X^TX$ squares the condition number, $\kappa(X^TX) = \kappa(X)^2$ (lecture 11). Small singular values of $X$ become tiny eigenvalues of $S$ that can round to zero or even negative, and their eigenvectors are lost to noise. The SVD works on $X$ directly and never forms the product.
- **Efficiency and memory.** Truncated and randomized SVD routines compute only the top $d$ singular triples, which for a wide matrix is far cheaper than the full $n\times n$ eigenproblem and never stores $S$.
- **Directness.** One call returns the components ($V$), the singular values (hence the explained variance) and the scores ($U'\Sigma'$). scikit-learn's `PCA` is an SVD under the hood.

## The t-SNE Objective

t-SNE [@van2008visualizing] replaces distances by probabilities, once in the high-dimensional space and once in the map, and moves the map's points until the two sets of probabilities agree as well as they can.

### High-dimensional similarities

For points $\mathbf{x}_1, \dots, \mathbf{x}_m$, the conditional probability that point $i$ would pick $j$ as its neighbour, under a Gaussian centred at $\mathbf{x}_i$ with bandwidth $\sigma_i$, is

$$
p_{j\vert i} = \frac{\exp(-\|\mathbf{x}_i - \mathbf{x}_j\|^2 / 2\sigma_i^2)}{\sum_{k \neq i} \exp(-\|\mathbf{x}_i - \mathbf{x}_k\|^2 / 2\sigma_i^2)},
\qquad p_{i\vert i} = 0.
$$

Nearby points receive most of the probability, far points almost none, and a point is not its own neighbour.

### Symmetrization

In general $p_{j\vert i} \neq p_{i\vert j}$: each point has its own bandwidth, and a point in a dense region "chooses" its neighbours differently from one in a sparse region. t-SNE combines the two conditionals into a single joint distribution over pairs,

$$
p_{ij} = \frac{p_{j\vert i} + p_{i\vert j}}{2m}.
$$

Dividing by $2m$ makes $\sum_{i \neq j} p_{ij} = 1$, so $P$ is one probability distribution over all pairs. It also guarantees $\sum_j p_{ij} > \frac{1}{2m}$ for every point $i$, so even an outlier contributes to the cost and is not simply ignored. Using this $p_{ij}$ is called *symmetric* t-SNE.

### Perplexity sets each $\sigma_i$

The bandwidth $\sigma_i$ is not a global constant. Each one is chosen so that the Gaussian neighbourhood of $\mathbf{x}_i$ contains a fixed *effective number of neighbours*, the **perplexity**. Let $P_i$ be the distribution $p_{\cdot\vert i}$ induced by $\sigma_i$ and let

$$
H(P_i) = -\sum_j p_{j\vert i} \log_2 p_{j\vert i}
$$

be its entropy. The perplexity of $P_i$ is

$$
\operatorname{Perp}(P_i) = 2^{H(P_i)},
$$

and $\sigma_i$ is found by binary search so that $\operatorname{Perp}(P_i)$ equals the user's chosen value. If the $k$ nearest neighbours of $\mathbf{x}_i$ were equally likely and everything else impossible, $H(P_i) = \log_2 k$ and the perplexity would be exactly $k$, hence "effective number of neighbours". Dense regions get a small $\sigma_i$, sparse regions a large one. Low perplexity (5--10) looks only at very local structure and the map can fragment into many small clumps; high perplexity (50--100) uses broader neighbourhoods and small clusters merge. It must be smaller than the number of points; scikit-learn's default is 30.

### Low-dimensional similarities

The map points $\mathbf{y}_1, \dots, \mathbf{y}_m \in \mathbb{R}^2$ get their own joint distribution, from a Student-t kernel with one degree of freedom instead of a Gaussian:

$$
q_{ij} = \frac{(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2)^{-1}}{\sum_{k \neq l} (1 + \|\mathbf{y}_k - \mathbf{y}_l\|^2)^{-1}},
\qquad q_{ii} = 0.
$$

### The Kullback--Leibler cost and its asymmetry

The map is fit by minimizing the Kullback--Leibler divergence of $Q$ from $P$,

$$
C = KL(P \| Q) = \sum_{i \neq j} p_{ij} \log \frac{p_{ij}}{q_{ij}}.
$$

$KL(P\|Q) \geq 0$, with equality only when $Q = P$. A perfect map would reproduce every pairwise similarity, which is usually impossible in 2-D, so what matters is *which* mismatches cost the most. The weights $p_{ij}$ decide:

- $p_{ij}$ large but $q_{ij}$ small (neighbours in high-D placed far apart in the map): a **large** penalty.
- $p_{ij}$ small but $q_{ij}$ large (distant points in high-D placed close in the map): $p_{ij}\log(p_{ij}/q_{ij})$ stays **small**, because it is multiplied by the small $p_{ij}$.

So t-SNE works hard to keep neighbours together and barely cares where non-neighbours land relative to one another. This asymmetry is the reason local structure is preserved while cluster sizes and the gaps between clusters are not, which the lecture's three-blobs figure shows directly.

### The gradient

Differentiating $C$ with respect to a map point gives (the derivation is in the paper's appendix)

$$
\frac{\partial C}{\partial \mathbf{y}_i} = 4\sum_{j \neq i}(p_{ij} - q_{ij})\,(\mathbf{y}_i - \mathbf{y}_j)\,\bigl(1 + \|\mathbf{y}_i - \mathbf{y}_j\|^2\bigr)^{-1}.
$$

Read it as a sum of springs: each other point $j$ pulls $\mathbf{y}_i$ toward itself when $p_{ij} > q_{ij}$ (the map underestimates their similarity) and pushes it away when $p_{ij} < q_{ij}$, with a strength that fades like $1/(1 + \text{distance}^2)$. The positions are updated by gradient descent with momentum, and implementations add two standard tricks: *early exaggeration* (multiplying all $p_{ij}$ by a constant for the first iterations, so clusters form before they are refined) and, for large $m$, the Barnes--Hut approximation that makes each iteration $O(m\log m)$ rather than $O(m^2)$.

### Why a Student-t in the map

In [ ]:
#| fig-align: center
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, t

x = np.linspace(-6, 6, 400)
gaussian_y = norm.pdf(x)        # mean 0, standard deviation 1
t_y = t.pdf(x, df=1)            # Student's t with one degree of freedom

plt.plot(x, gaussian_y, label='Gaussian (high-D similarity)')
plt.plot(x, t_y, linestyle='dashed', label="Student's t, 1 d.o.f. (low-D similarity)")

x0 = 3
plt.annotate(f'at a distance of {x0}:\nGaussian density {norm.pdf(x0):.4f}\n'
             f"Student's t density {t.pdf(x0, df=1):.3f}",
             xy=(x0, t.pdf(x0, df=1)), xytext=(3.2, 0.18), ha='center',
             arrowprops=dict(arrowstyle='->'))

plt.xlabel('distance between a pair of points')
plt.ylabel('probability density')
plt.title("Gaussian vs. Student's t-distribution")
plt.legend()
plt.show()

The two curves agree near zero, so close neighbours are treated alike. In the tails the Student's t keeps several times more mass: a pair that is moderately far apart in high-D can be placed *much* farther apart in the map and still match its (tiny) $p_{ij}$. That extra room is what relieves the **crowding problem**: there is far less area in 2-D than volume in 64-D to spread the non-neighbours into, and with a Gaussian in both spaces the moderately distant points would all be crushed toward the centre of the map.

### What this implies for reading a map

Everything the lecture says about t-SNE plots follows from the cost. There is no `transform` for new points because the $\mathbf{y}_i$ are free variables of one optimization, not the image of a function. Different seeds give different pictures because the cost is non-convex and the starting positions differ. Cluster sizes and inter-cluster gaps are not meaningful because the cost barely penalizes where non-neighbours are placed. And perplexity changes the picture because it changes $P$ itself. The only structure the objective actually protects is *who is near whom*.